# Accident detection


> DataSet-- https://universe.roboflow.com/dataset-tfm18/zihan-z36um/dataset/5

> Results-- "https://wandb.ai/ahmed-hossam-suez-canal-university/Accident_Severity_Detection/table?nw=nwuserahmedhossam"


## Libraries & Dataset

In [1]:
!pip install -qU roboflow ultralytics wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.9/207.9 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 36.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.2/27.2 MB 61.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 94.8 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.2.0 which is incompatible.


In [2]:
from kaggle_secrets import UserSecretsClient
import wandb

user_secrets = UserSecretsClient()
roboflow_api_key = user_secrets.get_secret("RoboFlow")
wandb_api_key = user_secrets.get_secret("WandB_SafeSpace")

wandb.login(key=wandb_api_key)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ahmed-hossam (ahmed-hossam-suez-canal-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [3]:
from roboflow import Roboflow

rf = Roboflow(api_key=roboflow_api_key)
project = rf.workspace("accident-and-nonaccident").project("accident-and-non-accident-label-image-dataset")
version = project.version(14)
dataset = version.download("yolo26")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Accident-and-Non-accident-label-Image-Dataset-14 in yolo26:: 100%|██████████| 6196/6196 [00:00<00:00, 7872.28it/s]


## Check Dataset Imbalance

In [7]:
import os

labels_dir = "/kaggle/working/Accident-and-Non-accident-label-Image-Dataset-14/train/labels"

class_counts = {}

for file in os.listdir(labels_dir):
    file_path = os.path.join(labels_dir, file)
    
    with open(file_path, "r") as f:
        lines = f.readlines()
        for line in lines:
            class_id = line.split()[0]
            class_counts[class_id] = class_counts.get(class_id, 0) + 1

print("Class counts:", class_counts)

max_class = max(class_counts.values())
min_class = min(class_counts.values())

print("Imbalance ratio:", max_class / min_class)

image_classes = {}

for file in os.listdir(labels_dir):
    file_path = os.path.join(labels_dir, file)
    
    with open(file_path, "r") as f:
        lines = f.readlines()
        classes_in_image = set(line.split()[0] for line in lines)
        
        for cls in classes_in_image:
            image_classes[cls] = image_classes.get(cls, 0) + 1

print("Image-level counts:", image_classes)

Class counts: {'0': 2313, '1': 1807}
Imbalance ratio: 1.2800221361372441
Image-level counts: {'0': 2092, '1': 892}


## Initializing the Run

In [ ]:
# ── W&B: Initialize run with full hyperparameter config ──────────────
EPOCHS = 60
IMGSZ  = 640
BATCH  = 16
MODEL  = "yolo26n.pt"
PROJECT = "Accident_Severity_Detection"
RUN_NAME = "v3_Accident_detection_AG"           # edit AG to your short name & start with v1 & edit the version number as you go !!!!

run = wandb.init(
    project=PROJECT,
    name=RUN_NAME,
    job_type="training",
    config = {
        "model":        MODEL,
        "pretrained":   True,
        
        # Core Training
        "epochs":          EPOCHS,
        "imgsz":           IMGSZ,
        "batch":           BATCH,
        
        # Optimizer & LR
        "optimizer":       "AdamW",
        "lr0":             0.003,
        "lrf":             0.01,
        "weight_decay":    0.0007,
        "warmup_epochs":   3.0,
        
        # Augmentation 
        "fliplr":          0.5,    # flipped
        "hsv_h":           0.015,  # hue
        "hsv_s":           0.7,    # saturation
        "hsv_v":           0.4,    # brightness/exposure
        "mosaic":          0.5,    
        "mixup":           0.1,   

        # Regularization 
        "label_smoothing": 0.05,

        # Training stability
        "patience": 10,               # early stopping
        "cos_lr": True,               # smoother LR decay

        # Dataset
        "dataset":         "accident-and-non-accident-label-image-dataset",
        "dataset_version": 14,
        "dataset_link":    "https://universe.roboflow.com/accident-and-nonaccident/accident-and-non-accident-label-image-dataset/dataset/14",
        "num_classes":     2,
}
)
print(f"W&B run started: {run.url}")


## Modeling

In [ ]:
from ultralytics import YOLO

cfg = wandb.config  # use values logged to W&B

model = YOLO(cfg.model)

results = model.train(
    data='/kaggle/working/Accident-and-Non-accident-label-Image-Dataset-14/data.yaml',

    # Core Training
    epochs=cfg.epochs,
    imgsz=cfg.imgsz,
    batch=cfg.batch,

    # Optimizer & LR 
    optimizer=cfg.optimizer,
    lr0=cfg.lr0,
    lrf=cfg.lrf,
    weight_decay=cfg.weight_decay,
    warmup_epochs=cfg.warmup_epochs,

    # Augmentation
    fliplr=cfg.fliplr,
    hsv_h=cfg.hsv_h,
    hsv_s=cfg.hsv_s,
    hsv_v=cfg.hsv_v,
    mosaic=cfg.mosaic,   
    mixup=cfg.mixup,

    # Regularization 
    label_smoothing = cfg.label_smoothing,

    # Training stability
    patience=cfg.patience,   
    cos_lr=cfg.cos_lr,   
    
    # Logging
    project=PROJECT,
    name=RUN_NAME,
    plots=True,
)

## Logging the Results

In [ ]:
# ── W&B: Log final validation metrics ─────────────────────────────────
metrics_dict = results.results_dict

final_metrics = {
    "final/precision":    metrics_dict.get("metrics/precision(B)", 0),
    "final/recall":       metrics_dict.get("metrics/recall(B)",    0),
    "final/mAP50":        metrics_dict.get("metrics/mAP50(B)",     0),
    "final/mAP50-95":     metrics_dict.get("metrics/mAP50-95(B)",  0),
    "final/fitness":      results.fitness,
}
wandb.log(final_metrics)

# Also write them as W&B summary so they show in the runs table
for k, v in final_metrics.items():
    wandb.run.summary[k] = v

print("Logged metrics:")
for k, v in final_metrics.items():
    print(f"  {k}: {v:.4f}")


In [ ]:
# ── W&B: Log training plots & validation images ───────────────────────
from pathlib import Path

save_dir = Path(results.save_dir)

plot_files = {
    "confusion_matrix":           save_dir / "confusion_matrix.png",
    "confusion_matrix_normalized": save_dir / "confusion_matrix_normalized.png",
    "BoxPR_curve":                   save_dir / "BoxPR_curve.png",
    "BoxF1_curve":                   save_dir / "BoxF1_curve.png",
    "BoxP_curve":                    save_dir / "BoxP_curve.png",
    "BoxR_curve":                    save_dir / "BoxR_curve.png",
    "results":                    save_dir / "results.png",
    "labels":                     save_dir / "labels.jpg"
}

wandb_images = {}
for name, path in plot_files.items():
    if path.exists():
        wandb_images[f"plots/{name}"] = wandb.Image(str(path), caption=name)
        print(f"  ✓ {name}")
    else:
        print(f"  ✗ {name} not found")

# Validation batch predictions (ground-truth vs predictions)
for img_path in sorted(save_dir.glob("val_batch*.jpg")):
    wandb_images[f"val_batches/{img_path.stem}"] = wandb.Image(
        str(img_path), caption=img_path.stem
    )
    print(f"  ✓ {img_path.stem}")

wandb.log(wandb_images)
print(f"\nLogged {len(wandb_images)} images/plots to W&B.")


In [ ]:
# ── W&B: Save best model as a versioned artifact ──────────────────────
best_pt = save_dir / "weights" / "best.pt"
last_pt = save_dir / "weights" / "last.pt"

artifact = wandb.Artifact(
    name="Accident_detector",
    type="model",
    description="YOLOv26n fine-tuned for Accident detection",
    metadata={
        "mAP50":     wandb.run.summary.get("final/mAP50"),
        "mAP50-95":  wandb.run.summary.get("final/mAP50-95"),
        "precision": wandb.run.summary.get("final/precision"),
        "recall":    wandb.run.summary.get("final/recall"),
        "epochs":    cfg.epochs,
        "imgsz":     cfg.imgsz,
        "dataset_link": cfg.dataset_link,
    }
)

if best_pt.exists():
    artifact.add_file(str(best_pt), name="best.pt")
if last_pt.exists():
    artifact.add_file(str(last_pt), name="last.pt")

wandb.log_artifact(artifact)
artifact.wait()  # ← block until artifact is fully logged on the W&B server
print(f"Model artifact logged: {artifact.name}:{artifact.version}")


In [ ]:
# Finish the WandB run
wandb.finish()